In [ ]:
import os
import time
import math
import json
import random
import pandas as pd
import numpy as np
import pylab as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
import jupyter_black

jupyter_black.load()

In [ ]:
ARTICLE_TYPE = "research-article"
SECTIONS = ["introduction", "methods", "results", "discussion"]
ALLOW_OTHER = False  # should papers with more than the required sections be used?
FULL_TEXT = True
BASELINE_NAME = "baseline_2025-06-26"
DATA_PATH = os.path.join("../data", BASELINE_NAME, "formatted")
SUBSET = "2022"

### comparing the length of different sections

In [ ]:
# load and filter df
df, count_full, count_filtered = utils.load_paper_df(
    DATA_PATH, ARTICLE_TYPE, SECTIONS, FULL_TEXT, ALLOW_OTHER, SUBSET
)

In [ ]:
for m in range(1, 13):
    print(df[df["date"].apply(lambda x: x.month == m)].shape)

In [ ]:
count_full

In [ ]:
count_filtered

In [ ]:
df = df[df["date"] >= "2010-1-01"]
len(df)

In [ ]:
df = df[df["date"] >= "2020-11-01"]
len(df)

In [ ]:
sections = ["abstract", "introduction", "methods", "results", "discussion", "full"]

In [ ]:
df_count = df[sections].copy()

In [ ]:
for sec in sections:
    df_count[sec] = list(map(lambda x: len(x.split()), df_count[sec]))

In [ ]:
df_count = pd.melt(df_count, ignore_index=False).reset_index()

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df_count, x="variable", y="value", whis=[5, 95])
ax.set_ylim(0, 5000)

#### Thoughts about normalization by lenght
it is not immediatly clear if and how normalization by lenght would work
- most basic would be to throw out outliers, see if remaining texts are enough (with some luck, the really long sections are from the same papers)
- try a control set with texts that are cropped to the same lenght (use random dropout or crop, maybe both)
- try normalization in a toy situation? simulate a toy situation (assume word frequency per paragraph to be at some percentage for humans, and base percentage of llm usage, then simulate how the occurrence numbers would change with number of paragraphs)

### Cropping sections to the same length
... but which lenght? Introduction is the shortest section, but the shortest introduction only has 19 words. Take the mean? Or median?

In [ ]:
df_count[df_count["variable"] == "abstract"]["value"].median()

In [ ]:
df_count[df_count["variable"] == "abstract"]["value"].mean()

In [ ]:
for sec in sections:
    print(
        f"min lenght {sec}: "
        + str(df_count[df_count["variable"] == sec]["value"].min())
    )

In [ ]:
count = 0
for abs in df["abstract"].values:
    if len(abs.split()) < 50:
        count += 1
count

In [ ]:
df_count[df_count["variable"] == "methods"]["value"].min()

In [ ]:
intro = df["introduction"].iloc[0]

In [ ]:
intro_words = intro.split()

In [ ]:
" ".join(intro_words[0:5])

In [ ]:
len(intro_words)

In [ ]:
random.seed(0)

In [ ]:
utils.crop_section("This is a very thorough test", 7)

In [ ]:
random.seed(0)
a = "This is a very thorough test"
list(map(lambda x: utils.crop_section(x, 3), [a, a, a, a, a, a, a, a, a]))

In [ ]:
list(map(lambda x: utils.crop_section(x, 3), df["introduction"]))

In [ ]:
utils.sample_section("This is a very thorough test", 3)

In [ ]:
random.seed(0)
a = "This is a very thorough test"
list(map(lambda x: utils.sample_section(x, 3), [a, a, a, a, a, a, a, a, a]))

In [ ]:
list(map(lambda x: utils.sample_section(x, 3), df["introduction"]))

In [ ]:
vectorizer = CountVectorizer(binary=True, min_df=1e-6)
X = vectorizer.fit_transform(df["introduction"].values)
X.shape

In [ ]:
cropped = list(map(lambda x: utils.crop_section(x, 700), df["introduction"]))
X_crop = vectorizer.fit_transform(cropped)
X_crop.shape

In [ ]:
sampled = list(map(lambda x: utils.sample_section(x, 700), df["introduction"]))
X_sample = vectorizer.fit_transform(sampled)
X_sample.shape